# 2.5 Lab: Research Assistant Agent

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

## Your API key

Some cells in this notebook call a model and need your own OpenAI API key. Paste it when asked. It is stored only in this notebook session, never saved to the file.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Prerequisites

Before this lab, work through:

- **0.2** for the virtual environment, `.env` file, and `OPENAI_API_KEY`.
- **2.1** for how tool calling works and what `bind_tools` does.
- **2.2** for writing `@tool` functions with honest docstrings and friendly errors (the calculator comes from there).
- **2.3** for the ReAct loop and `create_agent` with `system_prompt=`.
- **2.4** for `ToolErrorMiddleware`, `ToolRetryMiddleware`, and `ModelCallLimitMiddleware` (this lab wires all three into a real agent).

## Learning objectives

- Build a research assistant agent with `create_agent` using three tools: a web-search tool, a calculator, and a summariser.
- Choose a search provider deliberately: Tavily as the primary (API key, metered) with DuckDuckGo as a free fallback, and know the trade-offs of each.
- Enforce source honesty in the system prompt: cite only fetched URLs, disclose degraded mode when search fails.
- Compress long research sessions with `SummarizationMiddleware` and long tool outputs with a summariser tool.
- Read a real agent trace and diagnose the four classic research-agent failures: cost blowout, rate limits, stale results, and confident-but-sourceless answers.

## What you are building

A single-file program, `research_agent.py`, that takes a research question from the command line, searches the web, does any arithmetic the question needs, and returns a short briefing with inline source links. Along the way it demonstrates every reliability pattern from lesson 2.4 in one place: the search tool gets retries and a call budget, errors become observations, and the whole run has a model-call ceiling.

One note on providers before you install anything. Tavily is a purpose-built search API for agents: it returns clean extracted content instead of HTML soup, which keeps observations small and on-topic. It is metered (a free tier exists, no card required). DuckDuckGo is free and needs no key, but it is a scraped endpoint: rate limits and blocks are common, so treat it as the fallback, not the foundation. Brave Search sits between them (API key, free tier, privacy-respecting; a solid pick for EU learners who want a vendor outside the big clouds). If you want full data residency, a self-hosted SearxNG instance wrapped in `@tool` keeps every query inside your own infrastructure. This lab uses Tavily first and DuckDuckGo when no Tavily key is set, and says so in the code.

## Step 1: install and keys

In [ ]:
!python -m venv .venv && source .venv/bin/activate
!pip install "langchain==1.4.6" "langchain-core==1.6.12" \
!    "langchain-openai==1.7.2" "langchain-tavily==0.2.18" \
!    "langchain-community" "duckduckgo-search" "python-dotenv"
!printf 'OPENAI_API_KEY=sk-...\nTAVILY_API_KEY=tvly-...\n' > .env

The `TAVILY_API_KEY` line is optional. Without it, the lab falls back to DuckDuckGo automatically. Keep `.env` out of version control.

## Step 2: the search tool, with a fallback

In [ ]:
"""research_agent.py - a research assistant with search, calculator, and summariser.

Setup (run once): see Step 1.
Run: python research_agent.py "What is the current EU AI Act timeline for GPAI models?"
"""

import os

from dotenv import load_dotenv

from langchain_core.tools import tool

load_dotenv()


def _tavily_tool():
    from langchain_tavily import TavilySearch
    return TavilySearch(max_results=5, search_depth="advanced")


def _duckduckgo_tool():
    from langchain_community.tools import DuckDuckGoSearchResults
    return DuckDuckGoSearchResults(num_results=5)


if os.getenv("TAVILY_API_KEY"):
    web_search = _tavily_tool()
    web_search.description = (
        "Search the web for current facts. Returns extracted article content. "
        "Use when the question needs information newer than your training data, "
        "or when claims need a checkable source. Prefer 1-2 precise queries over many broad ones."
    )
    print("Using Tavily search (metered).")
else:
    web_search = _duckduckgo_tool()
    print("No TAVILY_API_KEY set: using DuckDuckGo (free, may be rate-limited).")

Two deliberate choices here. `max_results=5` keeps each observation small; a research agent that fetches 20 results per query pays for 20 in every subsequent model call, because observations stay in the history. And the description teaches query discipline ("prefer 1-2 precise queries"), because search cost is driven more by how often the model searches than by the per-search price.

## Step 3: calculator and summariser

The calculator is the one from lesson 2.2, reused unchanged. The summariser is new: a tool the agent can call when a search returns a wall of text it wants condensed before reasoning further.

In [ ]:
from langchain_openai import ChatOpenAI

_summariser = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)


@tool
def summarise(text: str, focus: str = "") -> str:
    """Condense a long text into 5-8 bullet points.

    Use this when a search result or tool output is too long to reason over
    directly. Keeps dates, numbers, and names verbatim; drops everything else.
    Args:
        text: The long text to condense.
        focus: Optional aspect to emphasise, e.g. 'timeline' or 'costs'.
    """
    prompt = (
        "Summarise the text below into 5-8 bullet points. "
        "Keep every date, number, and proper name exactly as written. "
        "Do not add information that is not in the text."
    )
    if focus:
        prompt += f" Emphasise: {focus}."
    return _summariser.invoke(
        [{"role": "user", "content": prompt + "\n\n---\n\n" + text[:12000]}]
    ).content

The 12,000-character cap is doing real work: it bounds the summariser's input cost regardless of how bloated a search result is. The "do not add information" instruction is load-bearing too; a summariser that invents facts poisons every later step. For whole-conversation compression (as opposed to one long output), the agent below also uses `SummarizationMiddleware`, which summarises old history before each model call once it passes a message threshold. Tool for single outputs, middleware for the running history.

## Step 4: assemble the agent with the full reliability stack

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    SummarizationMiddleware,
    ToolCallLimitMiddleware,
    ToolErrorMiddleware,
    ToolRetryMiddleware,
)

SYSTEM_PROMPT = (
    "You are a research assistant. Answer the user's question with a short "
    "briefing: 4-8 bullet points, then a Sources section listing every URL "
    "you actually fetched. Rules: cite ONLY URLs returned by the web_search "
    "tool; never invent links. If search fails or is rate-limited, say so "
    "plainly and answer from what you have, marked as unverified. Do the "
    "arithmetic yourself only for trivial sums; use the calculator tool for "
    "anything with more than two steps. Keep total searches under 5."
)

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[web_search, calculate, summarise],
    system_prompt=SYSTEM_PROMPT,
    middleware=[
        ToolErrorMiddleware(tools=[web_search]),          # search hiccups become observations
        ToolRetryMiddleware(
            tools=[web_search],
            max_retries=2,
            retry_on=(TimeoutError, ConnectionError),     # transient only, per lesson 2.4
        ),
        ToolCallLimitMiddleware(tool_name="web_search", run_limit=5),
        ModelCallLimitMiddleware(run_limit=25),
        SummarizationMiddleware(
            max_messages_before_summary=20,
            summary_model=ChatOpenAI(model="gpt-4o-mini", temperature=0.0),
        ),
    ],
)

Every piece of lesson 2.4 is here, scoped deliberately: the retry and error middleware apply only to the search tool (the calculator does not need retries, and the summariser's failures should surface), the search tool gets its own call budget of 5 per run, and the whole run is capped at 25 model calls. The summarisation model is the cheap one; never pay flagship prices to compress history.

## Step 5: run it and read the trace

In [ ]:
import sys

question = sys.argv[1] if len(sys.argv) > 1 else "What is the current EU AI Act timeline for GPAI models?"

result = agent.invoke({"messages": [{"role": "user", "content": question}]})

print("\n===== FINAL ANSWER =====\n")
print(result["messages"][-1].content)

print("\n===== TRACE (what actually happened) =====\n")
for m in result["messages"]:
    kind = type(m).__name__
    if kind == "AIMessage" and getattr(m, "tool_calls", None):
        for c in m.tool_calls:
            print(f"TOOL CALL  {c['name']}({str(c['args'])[:120]})")
    elif kind == "ToolMessage":
        print(f"OBSERVATION ({m.name}): {str(m.content)[:200]}...")

Run it twice: once with a factual question ("current EU AI Act timeline"), once with something the model could answer from memory ("explain gradient descent"). The second run should barely touch the search tool. If it searches anyway, the system prompt's "when to search" sentence needs strengthening. Then deliberately break things: unset the Tavily key and watch the DuckDuckGo fallback (slower, noisier); disconnect from the network and watch errors become observations and the agent disclose degraded mode. Labs are for breaking things on purpose.

## Step 6: the honesty checklist (make it yours)

Before you call this lab done, verify each of these by running the agent:

1. Every URL in the Sources section appears in a real `OBSERVATION` in the trace. No invented links.
2. With the network off, the final answer says search failed and marks claims as unverified, instead of writing a confident briefing anyway.
3. A question needing arithmetic ("a 19% VAT on EUR 1,240, split across 3 invoices") routes through the calculator tool, not mental math.
4. Five searches never happen: the `ToolCallLimitMiddleware` budget of 5 holds, and the agent wraps up with what it has.

## What can go wrong

- **Query fan-out explodes the bill.** This is the number one real-world research-agent failure: a few hundred user prompts triggering thousands of metered searches, because nothing capped searches per task. `ToolCallLimitMiddleware` and the "prefer 1-2 precise queries" sentence are the fix; prompt-only budgets do not hold.
- **DuckDuckGo rate-limits you mid-demo.** The scraped endpoint blocks aggressively. That is why it is the fallback. If your demo must be reliable, get the free Tavily key.
- **Stale index, fresh confidence.** Search indexes lag reality, especially for fast-moving topics like AI regulation. The agent reports what the index says, confidently. Teach it the phrase "as of the latest indexed sources" and check primary sources for anything load-bearing.
- **Snippet-only hallucinations.** When results are truncated, the model fills gaps with plausible inventions and cites the real URL anyway. The lab's rule (cite only fetched content, disclose degraded mode) exists because this failure looks exactly like success.
- **Summaries lose exact facts.** `SummarizationMiddleware` is lossy by design. Dates, figures, and names the final briefing needs should live in tool outputs the agent has already read, not only in the compressed history. If a briefing drops a date, check whether it was summarised away.
- **Search queries leak the question.** Every query goes to a third party, verbatim. For EU learners: a research question can contain personal or client data, which makes the search provider a data processor under GDPR. Prefer EU-hosted or self-hosted search for sensitive topics, and never put secrets in a research question.

## Check your understanding

**1. Why is Tavily the primary search tool in this lab and DuckDuckGo only the fallback?**
Tavily is a purpose-built agent search API: it returns clean extracted content, has predictable pricing, and behaves reliably. DuckDuckGo is free and keyless but scrapes an endpoint that rate-limits and blocks often. Reliability first, free second; the code falls back gracefully when no Tavily key is set.

**2. The lab scopes `ToolRetryMiddleware` and `ToolErrorMiddleware` to `tools=[web_search]`. Why not apply them to all three tools?**
The calculator's failures are deterministic (bad input), so retries would waste calls and error-as-data already covers it. The summariser's failures should surface rather than be silently retried. Scoping keeps each policy matched to the failure class of the tool, which is the whole point of lesson 2.4.

**3. What is the difference between the `summarise` tool and `SummarizationMiddleware`?**
The tool condenses one long output on demand, called by the agent when an observation is too big to reason over. The middleware compresses the whole conversation history automatically once it passes a message threshold, before each model call. One handles single bloated outputs; the other keeps long sessions affordable.

**4. Search is down and the agent writes a polished briefing with a Sources section anyway. What failed, and what is the fix?**
The honesty rules failed: the agent did not disclose degraded mode and may have invented citations. The fix is the system prompt's explicit rules (cite only fetched URLs; say plainly when search fails; mark claims unverified), verified by the lab's honesty checklist against the actual trace, not the final answer.

**5. A learner runs 200 research questions and the Tavily bill surprises them. Name two controls from this lab that bound search spend.**
`ToolCallLimitMiddleware(tool_name="web_search", run_limit=5)` caps searches per run in code, and the tool description's "prefer 1-2 precise queries" teaches the model to search sparingly. Together they bound the fan-out that causes real-world bill shock.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.